In [2]:
# !pip install scikeras

# # Uninstall the current scikit-learn version
# !pip uninstall scikit-learn -y

# # Install a compatible version of scikit-learn (e.g., 1.4.2)
# !pip install scikit-learn==1.4.2

# !pip install -q scikeras imbalanced-learn

!pip uninstall -y scikit-learn scikeras
!pip install scikit-learn==1.4.2 scikeras imbalanced-learn

Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 70.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.4.2 which is incompatible.


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from keras.callbacks import EarlyStopping, ReduceLROnPlateau

from scikeras.wrappers import KerasClassifier

from sklearn.preprocessing import LabelEncoder, StandardScaler,OneHotEncoder


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, accuracy_score, confusion_matrix,
    f1_score, fbeta_score,
    matthews_corrcoef, brier_score_loss, roc_auc_score
)

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.calibration import CalibrationDisplay
from sklearn.utils.class_weight import compute_class_weight

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,IterativeImputer

In [8]:
train_faults = pd.read_csv('training_faults_diagnostics.csv', low_memory=False)
train_faults.head()

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1,2015-02-21 10:47:13,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0,111,17,True,...,NaN,False,78.8,1023,True,NaN,0.0,3276.75,NaN,0.0
1,2,2015-02-21 11:34:34,NaN,unknown,unknown,unknown,11,629,12,True,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2015-02-21 11:35:31,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11,1807,2,False,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2015-02-21 11:35:33,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11,1807,2,True,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2015-02-21 11:39:41,NaN,22281684P01*22357957P01*22362082P01*,0USA13_13_0415_2238A,VOLVO,0,4364,17,False,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# # Convert timestamps to datetime objects
# train_faults['EventTimeStamp'] = pd.to_datetime(train_faults['EventTimeStamp'])
# train_faults['LocationTimeStamp'] = pd.to_datetime(train_faults['LocationTimeStamp'])

# train_faults['EventTimeStamp'] = train_faults['EventTimeStamp'].astype('int64') // 10**9
# train_faults['LocationTimeStamp'] = train_faults['LocationTimeStamp'].astype('int64') // 10**9

In [ ]:
train_faults.info()

In [ ]:
# Your target
target = "Derate_Target_2.0-0.001"

# Convert timestamps
for train_faults in [X]:

   train_faults["EventTimeStamp"] = pd.to_datetime(train_faults["EventTimeStamp"])
   train_faults["LocationTimeStamp"] = pd.to_datetime(train_faults["LocationTimeStamp"])

    # Hour / weekday
   train_faults["event_hour"] = train_faults["EventTimeStamp"].dt.hour
   train_faults["event_day"] = train_faults["EventTimeStamp"].dt.dayofweek

    # Unix timestamp
   train_faults["event_ts"] = train_faults["EventTimeStamp"].astype("int64") // 10**9

    # Sort by equipment + time
   train_faults.sort_values(
        by=["EquipmentID", "event_ts"],
        inplace=True
    )

    # Number of past records for equipment
  #  train_faults["equip_seen_count"] = (
  #       train_faults.groupby("EquipmentID")
  #       .cumcount()
  #   )

  #   # Time gap between events
  #  train_faults["time_gap"] = (
  #       train_faults.groupby("EquipmentID")["event_ts"]
  #       .diff()
  #       .fillna(0)
  #   )

# Drop raw timestamps
# X = X.drop(
#     columns=["EventTimeStamp", "LocationTimeStamp"]
# )

In [ ]:
train_faults.info()

In [5]:

target = 'Derate_Target_2.0-0.001'
features = train_faults.drop(['Derate_Target_4.0-2.0','Derate_Target_8.0-2.0','Derate_Target_12.0-2.0',target,'IsFullDerate'], axis = 1)

X = features
y = train_faults[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=321)

In [ ]:
train_faults["equip_seen_count"] = X_train.groupby("EquipmentID").cumcount()
train_faults["time_gap"] = X_train.groupby("EquipmentID")["event_ts"].diff()

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns

In [ ]:
drop_cols = num_cols[X_train[num_cols].isnull().mean() > 0.8]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols, errors='ignore')

In [ ]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns

# split numeric further
low_missing = X_train[num_cols].columns[X_train[num_cols].isnull().mean() <= 0.4]
mid_missing = X_train[num_cols].columns[(X_train[num_cols].isnull().mean() > 0.4) &
                                   (X_train[num_cols].isnull().mean() <= 0.8)]

# #Categorical

# cat_imputer = SimpleImputer(strategy="most_frequent")

# X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
# X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# # Low missing numeric
# low_num_imputer = SimpleImputer(strategy="median")

# X_train[low_missing] = low_num_imputer.fit_transform(X_train[low_missing])
# X_test[low_missing] = low_num_imputer.transform(X_test[low_missing])


In [ ]:
# # Mid missing numeric
# # scaler = StandardScaler()
# # scaled_data = scaler.fit_transform(df[mid_missing])

# #Iterativeimputer uses multiple features to estimate missing values, capturing complex relationships between columns
# iter_imputer = IterativeImputer(max_iter=3,
#     tol=1e-1,random_state=30)

# # df[mid_missing] = iter_imputer.fit_transform(scaled_data)
# # training_faults_diagnostics_df = df


# X_train[mid_missing] = iter_imputer.fit_transform(X_train[mid_missing])
# X_test[mid_missing] = iter_imputer.transform(X_test[mid_missing])


In [ ]:
low_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
mid_num_pipeline = Pipeline(steps=[
    ("imputer", IterativeImputer(
        max_iter=3,
        tol=1e-1,
        random_state=30
    )),
    ("scaler", StandardScaler())
])

In [ ]:
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
ct = ColumnTransformer(
    transformers=[
        ("low_num", low_num_pipeline, low_missing),
        ("mid_num", mid_num_pipeline, mid_missing),
        ("cat", cat_pipeline, cat_cols)
    ],
    remainder="drop"
)

In [ ]:
X_train = ct.fit_transform(X_train)
X_test  = ct.transform(X_test)

Finally, we'll build a model, using the Sequential model class.



In [ ]:
# Function to create the Keras model for SciKeras
n_features = X_train.shape[1]
def create_model():
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    # model.add(tf.keras.layers.Dense(128, activation='relu')) #relu for hidden layers, Relu outputs zero for negative values and keeps positive values unchanged
    # model.add(tf.keras.layers.Dense(128, activation='relu'))
    # model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(64, activation='relu'))
    model.add(tf.keras.layers.Dense(32, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid')) # sigmoid for binary classification, probabilty that engine will enter derate condition
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

# Keras model with SciKeras wrapper
model = KerasClassifier(
    model=create_model,
    epochs=10,   # 100 complete passthrough of training data
    batch_size=512 #Keras split your training data into 2604 mini-batches.
    #callbacks=[es]
)


In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

We are training a model to predict derate probability, using Adam to learn efficiently, binary crossentropy to measure error, and accuracy to evaluate performance.”


In [ ]:
print(type(X_train))

In [ ]:
# Create and fit model pipeline
# pipe = Pipeline(
#     steps=[
#         #('scaler', StandardScaler(with_mean=False)), #only scales variance
#         ('model', model)
#     ]
# ).fit(X_train, y_train)

model.fit(X_train,y_train, class_weight=class_weights)


In [ ]:
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

In [ ]:
print(confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
))

In [ ]:
print(confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)
)

In [ ]:
print(classification_report(
    y_true=y_train,
    y_pred=y_pred_train
))
print(classification_report(
    y_true=y_test,
    y_pred=y_pred_test
))

In [ ]:
cmk = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)


TN, FP, FN, TP = cmk.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

In [ ]:
test_faults = pd.read_csv('testing_faults_diagnostics.csv', low_memory=False)
test_faults.head()

In [ ]:
test_faults['EventTimeStamp'] = pd.to_datetime(test_faults['EventTimeStamp'])
test_faults['LocationTimeStamp'] = pd.to_datetime(test_faults['LocationTimeStamp'])

# test_faults['EventTimeStamp'] = test_faults['EventTimeStamp'].astype('int64') // 10**9
# test_faults['LocationTimeStamp'] = test_faults['LocationTimeStamp'].astype('int64') // 10**9

# Timestamp features
# test_faults["EventTimeStamp"] = pd.to_datetime(
#     test_faults["EventTimeStamp"]
# )

# test_faults["LocationTimeStamp"] = pd.to_datetime(
#     test_faults["LocationTimeStamp"]
# )

test_faults["event_hour"] = (test_faults["EventTimeStamp"].dt.hour)

test_faults["event_day"] = (test_faults["EventTimeStamp"].dt.dayofweek)

test_faults["event_ts"] = (test_faults["EventTimeStamp"].astype("int64") // 10**9)

test_faults.sort_values(by=["EquipmentID", "event_ts"],inplace=True)

test_faults["equip_seen_count"] = (test_faults.groupby("EquipmentID").cumcount())

test_faults["time_gap"] =  test_faults.groupby("EquipmentID")["event_ts"].diff().fillna(0)


In [ ]:
target = 'Derate_Target_2.0-0.001'

y_test_unseen = test_faults[target]
test_faults = test_faults.drop(['Derate_Target_4.0-2.0','Derate_Target_8.0-2.0','Derate_Target_12.0-2.0',target,'IsFullDerate'], axis = 1)

In [ ]:
y_unseen = ct.transform(test_faults)

In [ ]:
y_pred_unseen = model.predict(y_unseen)

In [ ]:
cm = confusion_matrix(y_test_unseen, y_pred_unseen)

TN, FP, FN, TP = cm.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

In [ ]:
print(confusion_matrix(
    y_true=y_train,
    y_pred=y_pred_train
))
print(confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
))

In [ ]:
cmk_class = confusion_matrix(
    y_true=y_test,
    y_pred=y_pred_test
)


TN, FP, FN, TP = cmk_class.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

 **Adjust the Threshold**

First, we'll split into a train and validation set so we can adjust various hyperparameters, and then a test set, so that we can see how well our model performs on unseen data.

In [35]:
# Read the false diagnostics data
#low_memory =False ensures pandas reads the file in one go and avoids mixed-type inference issues

train_faults = pd.read_csv('training_faults_diagnostics.csv', low_memory=False)
train_faults.head()

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1,2015-02-21 10:47:13,Low (Severity Low) Engine Coolant Level,unknown,unknown,unknown,0,111,17,True,...,NaN,False,78.8,1023,True,NaN,0.0,3276.75,NaN,0.0
1,2,2015-02-21 11:34:34,NaN,unknown,unknown,unknown,11,629,12,True,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
2,3,2015-02-21 11:35:31,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11,1807,2,False,...,NaN,NaN,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2015-02-21 11:35:33,Incorrect Data Steering Wheel Angle,unknown,unknown,unknown,11,1807,2,True,...,NaN,True,NaN,1279,NaN,NaN,NaN,NaN,NaN,NaN
4,5,2015-02-21 11:39:41,NaN,22281684P01*22357957P01*22362082P01*,0USA13_13_0415_2238A,VOLVO,0,4364,17,False,...,NaN,NaN,NaN,16639,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
train_faults.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1058069 entries, 0 to 1058068
Data columns (total 48 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   RecordID                   1058069 non-null  int64  
 1   EventTimeStamp             1058069 non-null  object 
 2   eventDescription           1002335 non-null  object 
 3   ecuSoftwareVersion         831493 non-null   object 
 4   ecuModel                   1002466 non-null  object 
 5   ecuMake                    1002466 non-null  object 
 6   ecuSource                  1058069 non-null  int64  
 7   spn                        1058069 non-null  int64  
 8   fmi                        1058069 non-null  int64  
 9   active                     1058069 non-null  bool   
 10  activeTransitionCount      1058069 non-null  int64  
 11  EquipmentID                1058069 non-null  object 
 12  MCTNumber                  1058069 non-null  int64  
 13  Latitude    

In [37]:

target = "Derate_Target_4.0-2.0"

# Convert timestamps to datetime
train_faults["EventTimeStamp"] = pd.to_datetime(train_faults["EventTimeStamp"])
train_faults["LocationTimeStamp"] = pd.to_datetime(train_faults["LocationTimeStamp"])

# Extract Hour/weekday to check if faults happen on certain times
train_faults["event_hour"] = train_faults["EventTimeStamp"].dt.hour
train_faults["event_day"] = train_faults["EventTimeStamp"].dt.dayofweek

# Convert datetime into numerical so the model can understand
train_faults["event_ts"] = train_faults["EventTimeStamp"].astype("int64") // 10**9

# Arrange data in correct time order
train_faults.sort_values(
    by=["EquipmentID", "event_ts"],
    inplace=True
)

# Check number of  records for equipment
train_faults["equip_seen_count"] = (
    train_faults.groupby("EquipmentID")
    .cumcount() #counts no.of records
)

# Check time gap between events to see if any irregular events
train_faults["time_gap"] = (
    train_faults.groupby("EquipmentID")["event_ts"]
    .diff()
    .fillna(0) #while checking the diff of first record it will be NaN so filling it with 0
)

# Check recent faults history for each equipment
train_faults["fault_rate_5"] = (
    train_faults.groupby("EquipmentID")[target]
    .shift() #only use past information
    .rolling(5, min_periods=1)
    .mean()
    .fillna(0)
)

# # Drop raw timestamps
# # X = X.drop(
# #     columns=["EventTimeStamp", "LocationTimeStamp"]
# )

In [38]:
#Define Target variable and Drop Unnecessary columns
target = 'Derate_Target_4.0-2.0'
features = train_faults.drop(['Derate_Target_2.0-0.001','Derate_Target_8.0-2.0','Derate_Target_12.0-2.0',target,'IsFullDerate'], axis = 1)

X = features
y = train_faults[target]

In [39]:
#Split data into train test data
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify = y, random_state = 321, train_size = 0.7)
#Further split training data into training and validation set to tune model performance
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, stratify = y_train, random_state = 321, train_size = 0.6/0.8)

In [40]:
#Identiy numerical and categoricalcolumns
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns

In [41]:
#Drop columns if missing values are above 80 percent
drop_cols = num_cols[X_train[num_cols].isnull().mean() > 0.8]

X_train = X_train.drop(columns=drop_cols)
X_test = X_test.drop(columns=drop_cols, errors='ignore')
X_val = X_val.drop(columns=drop_cols, errors="ignore")

In [42]:
#Redefining numerical and categorical columns after preprocessing
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "bool"]).columns

# split numeric further
low_missing = X_train[num_cols].columns[X_train[num_cols].isnull().mean() <= 0.4]
mid_missing = X_train[num_cols].columns[(X_train[num_cols].isnull().mean() > 0.4) &
                                   (X_train[num_cols].isnull().mean() <= 0.8)]



In [43]:
# pipeline for low numerical missing features
low_num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) #Standard Scalar transforms data to mean 0 and std=1
])

In [44]:
# Pipeline for medium numerical missing values
#Iterative imputer predicts missing values using other feature relationships
mid_num_pipeline = Pipeline(steps=[
    ("imputer", IterativeImputer(
        max_iter=3,
        tol=1e-1,
        random_state=30
    )),
    ("scaler", StandardScaler())
])

In [45]:
# Pipeline for categorical missing values
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=50)) # creates binary columns min_frequency groups rare columns and reduces dummy columns, keeps only categories that appear atleast 50 times in data
])

In [46]:
#combine all preprocessing into single column transformer
process = ColumnTransformer(
    transformers=[
        ("low_num", low_num_pipeline, low_missing),
        ("mid_num", mid_num_pipeline, mid_missing),
        ("cat", cat_pipeline, cat_cols)
    ],
    remainder="drop"
)

In [47]:
#Fit and transform on training data learns patterns from data and applies transformation
X_train = process.fit_transform(X_train)
#Applies learning patterns without learning new
X_val   = process.transform(X_val)
X_test  = process.transform(X_test)

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [48]:
X_train.shape[1]

1553

In [49]:

n_features = X_train.shape[1] #get number of input features
def create_model():
    model_threshold = tf.keras.Sequential()
    model_threshold.add(tf.keras.layers.InputLayer(shape=(n_features,)))
    # model.add(tf.keras.layers.Dense(128, activation='relu')) #relu for hidden layers, Relu outputs zero for negative values and keeps positive values unchanged
    # model.add(tf.keras.layers.Dense(128, activation='relu'))
    # model.add(tf.keras.layers.Dense(128, activation='relu'))
    model_threshold.add(tf.keras.layers.Dense(64, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    model_threshold.add(tf.keras.layers.Dropout(0.2)) #20 % off neurons randomly  turnedoff in the layer, so the model canot heavily depend on specific neurons
    model_threshold.add(tf.keras.layers.Dense(32, activation='relu',kernel_regularizer=tf.keras.regularizers.l2(0.001)))
    model_threshold.add(tf.keras.layers.Dropout(0.2))
    model_threshold.add(tf.keras.layers.Dense(1, activation='sigmoid')) # sigmoid for binary classification, probabilty that engine will enter derate condition
    model_threshold.compile(optimizer='adam',loss='binary_crossentropy',metrics=[tf.keras.metrics.AUC(name='auc')])
    return model_threshold

es = EarlyStopping(
    monitor='val_loss',        # monitor validation loss on each epoch, lower validation loss is good
    patience=4,
    restore_best_weights=True,
    mode='min'                 # val_loss should go DOWN
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,      # halve the learning rate when stuck
    patience=3,      # after 3 epochs of no improvement
    min_lr=1e-6
)

# Keras model with SciKeras wrapper
model_threshold = KerasClassifier(
    model=create_model,
    epochs=20,
    batch_size=256  #The model trains on 256 rows at a time and then update its weights
    )

In [50]:
# classes = np.unique(y_train)
# weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
# class_weights = dict(zip(classes, weights))

In [51]:
#Balance classes for imbalnce classsifications
class_weights = {
    0: 1,
    1: 30, # mistakes on class 1 are 30 times more important than class0
}

In [52]:
#Fit the model
model_threshold.fit(X_train, y_train, validation_data=(X_val, y_val),class_weight=class_weights, callbacks=[es, reduce_lr])

Epoch 1/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 39s 16ms/step - auc: 0.9058 - loss: 0.0507 - val_auc: 0.9685 - val_loss: 0.0159
Epoch 2/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 33s 15ms/step - auc: 0.9253 - loss: 0.0297 - val_auc: 0.9121 - val_loss: 0.0098
Epoch 3/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 35s 16ms/step - auc: 0.9237 - loss: 0.0287 - val_auc: 0.9785 - val_loss: 0.0102
Epoch 4/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 40s 16ms/step - auc: 0.9148 - loss: 0.0282 - val_auc: 0.9518 - val_loss: 0.0092
Epoch 5/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 34s 16ms/step - auc: 0.9456 - loss: 0.0260 - val_auc: 0.9538 - val_loss: 0.0090
Epoch 6/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 43s 16ms/step - auc: 0.9322 - loss: 0.0262 - val_auc: 0.9161 - val_loss: 0.0070
Epoch 7/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 39s 16ms/step - auc: 0.9166 - loss: 0.0275 - val_auc: 0.9886 - val_loss: 0.0130
Epoch 8/20
2170/2170 ━━━━━━━━━━━━━━━━━━━━ 34s 16ms/step - auc: 0.9353 - loss: 0.0262 - val_auc: 0.9781 - val_loss: 0.0093
Epoch 9/20
2170/2170 ━━━

KerasClassifier(
	model=<function create_model at 0x78419406e0c0>
	build_fn=None
	warm_start=False
	random_state=None
	optimizer=rmsprop
	loss=None
	metrics=None
	batch_size=256
	validation_batch_size=None
	verbose=1
	callbacks=None
	validation_split=0.0
	shuffle=True
	run_eagerly=False
	epochs=20
	class_weight=None
)

In [53]:
#make predictions on test data
y_pred = model_threshold.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
#MCC is balanced metric for imbalanced classification
print(f'MCC: {matthews_corrcoef(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

1240/1240 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step
Accuracy: 0.9993258165023738
MCC: 0.43031158998546837
[[317137    177]
 [    37     70]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    317314
           1       0.28      0.65      0.40       107

    accuracy                           1.00    317421
   macro avg       0.64      0.83      0.70    317421
weighted avg       1.00      1.00      1.00    317421



In [54]:
#Predict probabilities on validation data
#instead of predicting 0 or 1 the model outputs the probability that each sample belongs to class 1
y_val_pred_proba = model_threshold.predict_proba(X_val)[:,1] #[:,1] select the probability of class 1
y_val_pred_proba

724/724 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step


array([0.00041964, 0.00466489, 0.00222259, ..., 0.00013507, 0.00027684,
       0.000584  ], dtype=float32)

In [55]:
#Generate candidate thresholds
candidate_thresholds = np.arange(start = 0.1, stop = 0.925, step = 0.01)
thresholds = pd.DataFrame({'threshold': candidate_thresholds})

#calculate savings
def profit_score(threshold):
    y_pred = (y_val_pred_proba > threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val,y_pred).ravel()
    return (tp * 4000) - (fp * 500)

thresholds['money_saved'] = thresholds['threshold'].apply(profit_score)
thresholds.sort_values('money_saved', ascending = False).head(20)

,threshold,money_saved
63,0.73,112500
62,0.72,109500
64,0.74,109000
61,0.71,108500
65,0.75,108000
45,0.55,107000
44,0.54,107000
43,0.53,107000
40,0.50,107000
41,0.51,107000


In [56]:
threshold = 0.73
y_pred_proba = model_threshold.predict_proba(X_test)[:,1]

y_pred = y_pred_proba > threshold
print(f'Accuracy: {accuracy_score(y_test, y_pred)}')
print(f'MCC: {matthews_corrcoef(y_test, y_pred)}')
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))



1240/1240 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step
Accuracy: 0.999357320404132
MCC: 0.4258956294490944
[[317150    164]
 [    40     67]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    317314
           1       0.29      0.63      0.40       107

    accuracy                           1.00    317421
   macro avg       0.64      0.81      0.70    317421
weighted avg       1.00      1.00      1.00    317421



In [57]:
cm = confusion_matrix(y_test, y_pred)

TN, FP, FN, TP = cm.ravel()

print("True Negatives (TN):", TN)
print("False Positives (FP):", FP)
print("False Negatives (FN):", FN)
print("True Positives (TP):", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

True Negatives (TN): 317150
False Positives (FP): 164
False Negatives (FN): 40
True Positives (TP): 67
Money saved:186000


Looks like model is producing zero probabilities for each sample and looks like threshold tuning is ineffective.

**Now let's see how model perform on unseen data**

In [58]:
#Read unseen data
test_faults = pd.read_csv('testing_faults_diagnostics.csv', low_memory=False)
test_faults.head()

,RecordID,EventTimeStamp,eventDescription,ecuSoftwareVersion,ecuModel,ecuMake,ecuSource,spn,fmi,active,...,FuelTemperature,IgnStatus,IntakeManifoldTemperature,LampStatus,ParkingBrake,ServiceDistance,Speed,SwitchedBatteryVoltage,Throttle,TurboBoostPressure
0,1100564,2019-01-01 00:24:10,Low (Severity Medium) Engine Coolant Level,PC4__1284P4C_6*,MX16U13D13,PCAR,0,111,18,True,...,60.8,True,59.0,17407,True,NaN,0.00000,NaN,0.0,0.58
1,1100565,2019-01-01 00:36:08,Low (Severity Medium) Engine Coolant Level,PC4__1284P4C_6*,MX16U13D13,PCAR,0,111,18,False,...,NaN,NaN,NaN,17407,NaN,NaN,NaN,NaN,NaN,NaN
2,1100566,2019-01-01 01:54:35,Low (Severity Medium) Engine Coolant Level,05317106*04150360*061416163421*09401361*G1*BDR*,6X1u13D1500000000,CMMNS,0,111,18,True,...,NaN,True,87.8,18431,True,NaN,0.00000,NaN,100.0,0.00
3,1100567,2019-01-01 02:08:21,Low (Severity Medium) Engine Coolant Level,05317106*04150360*061416163421*09401361*G1*BDR*,6X1u13D1500000000,CMMNS,0,111,18,False,...,NaN,NaN,NaN,17407,NaN,NaN,NaN,NaN,NaN,NaN
4,1100568,2019-01-01 03:06:52,Not Reporting Data Front Operator Wiper Switch,NaN,NaN,NaN,49,2863,7,True,...,NaN,True,105.8,255,False,NaN,66.89449,NaN,100.0,17.11


In [59]:
equipment_ids = test_faults["EquipmentID"].copy()

# Convert timestamps into datetime format
test_faults["EventTimeStamp"] = pd.to_datetime(test_faults["EventTimeStamp"])
test_faults["LocationTimeStamp"] = pd.to_datetime(test_faults["LocationTimeStamp"])

# Extract hour/dayofweek features
test_faults["event_hour"] = test_faults["EventTimeStamp"].dt.hour
test_faults["event_day"] = test_faults["EventTimeStamp"].dt.dayofweek

#Convert timestamp into numeric format
test_faults["event_ts"] = test_faults["EventTimeStamp"].astype("int64") // 10**9


# Sort data
test_faults = test_faults.sort_values(["EquipmentID", "event_ts"])

# Equipment history, track number of past records
test_faults["equip_seen_count"] = (
    test_faults.groupby("EquipmentID")
    .cumcount()
)

# calculate time gap between consecutive events
test_faults["time_gap"] = (
    test_faults.groupby("EquipmentID")["event_ts"]
    .diff()
    .fillna(0)
)


# Since true target is unknown,initialize rolling history with 0
test_faults["fault_rate_5"] = 0

# target column
target = "Derate_Target_2.0-0.001"

y_test_unseen = test_faults[target]




In [60]:
test_faults.info()

<class 'pandas.core.frame.DataFrame'>
Index: 129266 entries, 1147 to 93964
Data columns (total 54 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   RecordID                   129266 non-null  int64         
 1   EventTimeStamp             129266 non-null  datetime64[ns]
 2   eventDescription           124155 non-null  object        
 3   ecuSoftwareVersion         59792 non-null   object        
 4   ecuModel                   120111 non-null  object        
 5   ecuMake                    120111 non-null  object        
 6   ecuSource                  129266 non-null  int64         
 7   spn                        129266 non-null  int64         
 8   fmi                        129266 non-null  int64         
 9   active                     129266 non-null  bool          
 10  activeTransitionCount      129266 non-null  int64         
 11  EquipmentID                129266 non-null  int64      

In [61]:
#drop the columns as in training data
target = 'Derate_Target_4.0-2.0'

y_test_unseen = test_faults[target]
test_faults = test_faults.drop(['Derate_Target_2.0-0.001','Derate_Target_8.0-2.0','Derate_Target_12.0-2.0',target,'IsFullDerate','EventTimeStamp','LocationTimeStamp'], axis = 1)


In [62]:
y_test_unseen.shape

(129266,)

In [63]:
#use same process pipeline as used in training data
X_unseen = process.transform(test_faults)

In [64]:
#predict probabilities on unseen data
y_prob = model_threshold.predict_proba(X_unseen)[:,1]

505/505 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step


In [65]:
#convert probability scores into binary predictions
threshold = 0.73
unseen_y_pred = (y_prob > threshold).astype(int)

In [66]:
#cm = confusion_matrix(y_test_unseen, unseen_y_pred)
cm = confusion_matrix(y_test_unseen, unseen_y_pred)

TN, FP, FN, TP = cm.ravel()

print("TN:", TN)
print("FP:", FP)
print("FN:", FN)
print("TP:", TP)

print(f'Money saved:{TP * 4000 - FP * 500}')

TN: 129199
FP: 0
FN: 67
TP: 0
Money saved:0
